# Stage 3: Evaluation — Base vs SFT on Held-Out Test Set

**Environment:** Kaggle GPU (P100/T4)  
**Estimated time:** ~30–45 minutes  
**Input:** Test set + SFT adapter from Stage 1  
**Output:** `eval_results.json` with base & SFT responses for every test prompt

This notebook generates responses from both the **base Mistral 7B** and the
**SFT-tuned** model on the held-out test set, then saves them for offline
metrics computation (ROUGE-L, win-rate, etc.).  
DPO is excluded — it suffered mode collapse during training.

## 1. Install Dependencies

In [ ]:
!pip install -q transformers>=4.36.0 peft>=0.7.0 bitsandbytes>=0.41.0 accelerate>=0.25.0 datasets>=2.16.0

## 2. Verify Data Paths
Run this first to confirm exact paths on Kaggle.

In [ ]:
import os
print("=== All files in /kaggle/input/ ===")
for root, dirs, files in os.walk('/kaggle/input/'):
    for f in files:
        print(os.path.join(root, f))

## 3. Imports & Configuration
**IMPORTANT:** Update `SFT_ADAPTER_DIR` and `TEST_FILE` below if the paths from cell 2 differ.

In [ ]:
import os
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Disable W&B
os.environ["WANDB_DISABLED"] = "true"

# ── Paths ──
BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
SFT_ADAPTER_DIR = "/kaggle/input/datasets/aadityae/sft-adapter"
TEST_FILE = "/kaggle/input/datasets/aadityae/hr-preference-data/test.json"
OUTPUT_FILE = "/kaggle/working/eval_results.json"

# ── Quantization ──
LOAD_IN_4BIT = True
BNB_4BIT_QUANT_TYPE = "nf4"
BNB_4BIT_COMPUTE_DTYPE = torch.float16  # float16, NOT bfloat16
BNB_4BIT_USE_DOUBLE_QUANT = True

# ── Generation ──
MAX_NEW_TOKENS = 300
TEMPERATURE = 0.7
DO_SAMPLE = True

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 4. Load Test Set

In [ ]:
with open(TEST_FILE, "r", encoding="utf-8") as f:
    first_char = f.read(1)
    f.seek(0)
    if first_char == "[":
        # JSON array format
        test_data = json.load(f)
    else:
        # JSONL format (one JSON object per line) — Kaggle dataset uses this
        test_data = [json.loads(line) for line in f if line.strip()]

prompts = [item["prompt"] for item in test_data]
chosen_refs = [item["chosen"] for item in test_data]

print(f"Loaded {len(test_data)} test examples")
print(f"Sample prompt: {prompts[0][:100]}...")

## 5. Load Base Model (4-bit Quantized)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=LOAD_IN_4BIT,
    bnb_4bit_quant_type=BNB_4BIT_QUANT_TYPE,
    bnb_4bit_compute_dtype=BNB_4BIT_COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=BNB_4BIT_USE_DOUBLE_QUANT,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Base model loaded: {BASE_MODEL}")
print(f"Memory footprint: {base_model.get_memory_footprint() / 1e9:.2f} GB")

## 6. Generate Base Model Responses

In [ ]:
def generate_responses(model, tokenizer, prompts, label="model"):
    """Generate a response for each prompt using [INST] format."""
    responses = []
    total = len(prompts)
    for i, prompt in enumerate(prompts):
        formatted = f"[INST] {prompt} [/INST]"
        inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=TEMPERATURE,
                do_sample=DO_SAMPLE,
            )

        # Decode only the generated tokens (skip the input)
        generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
        response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        responses.append(response)

        if (i + 1) % 10 == 0 or (i + 1) == total:
            print(f"  [{label}] {i + 1}/{total} done")

    return responses

In [ ]:
print("Generating base model responses...")
base_responses = generate_responses(base_model, tokenizer, prompts, label="base")
print(f"\nDone! Generated {len(base_responses)} base responses.")

In [ ]:
# Sanity check — print a sample base response
print("=" * 60)
print("SAMPLE BASE RESPONSE")
print("=" * 60)
print(f"Prompt:   {prompts[0]}")
print(f"\nResponse: {base_responses[0][:500]}")
print("=" * 60)

## 7. Load SFT Adapter & Generate SFT Responses

In [ ]:
# Load SFT adapter on top of base model
sft_model = PeftModel.from_pretrained(base_model, SFT_ADAPTER_DIR, is_trainable=False)

print(f"SFT adapter loaded from: {SFT_ADAPTER_DIR}")
print(f"Memory footprint: {sft_model.get_memory_footprint() / 1e9:.2f} GB")

In [ ]:
print("Generating SFT model responses...")
sft_responses = generate_responses(sft_model, tokenizer, prompts, label="sft")
print(f"\nDone! Generated {len(sft_responses)} SFT responses.")

In [ ]:
# Sanity check — print a sample SFT response
print("=" * 60)
print("SAMPLE SFT RESPONSE")
print("=" * 60)
print(f"Prompt:   {prompts[0]}")
print(f"\nResponse: {sft_responses[0][:500]}")
print("=" * 60)

## 8. Save Results

In [ ]:
results = []
for i in range(len(prompts)):
    results.append({
        "prompt": prompts[i],
        "chosen": chosen_refs[i],
        "base_response": base_responses[i],
        "sft_response": sft_responses[i],
    })

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Saved {len(results)} results to {OUTPUT_FILE}")
print(f"File size: {os.path.getsize(OUTPUT_FILE) / 1024:.1f} KB")

## 9. Zip for Download

In [ ]:
!zip /kaggle/working/eval_results.zip /kaggle/working/eval_results.json
print("\nZipped! Go to Save Version -> Output tab to download.")

## 10. Quick Summary Stats

In [ ]:
# Print average response lengths as a basic sanity check
avg_base = sum(len(r) for r in base_responses) / len(base_responses)
avg_sft = sum(len(r) for r in sft_responses) / len(sft_responses)
avg_ref = sum(len(r) for r in chosen_refs) / len(chosen_refs)

print(f"Average response length (chars):")
print(f"  Reference (chosen): {avg_ref:.0f}")
print(f"  Base model:         {avg_base:.0f}")
print(f"  SFT model:          {avg_sft:.0f}")